# Task 1II - Customer Segmentation using Clustering

### Objective
To divide customers into groups based on their purchasing behaviour using the K-Means clustering algorithm.

**Libraries used:** Pandas, NumPy, Matplotlib, Scikit-learn

**Dataset:** Olist Brazilian E-Commerce Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# loading the datasets
orders = pd.read_csv("olist_data/olist_orders_dataset.csv")
payments = pd.read_csv("olist_data/olist_order_payments_dataset.csv")
items = pd.read_csv("olist_data/olist_order_items_dataset.csv")

print("Orders:", orders.shape)
print("Payments:", payments.shape)
print("Items:", items.shape)

## 1. Checking the customer information

In [ ]:
print("Number of unique customers:", orders["customer_id"].nunique())
print("Number of orders:", orders["order_id"].nunique())

orders.head()

## 2. Creating customer-level features

In [ ]:
# total payment for each order
payment_order = payments.groupby("order_id")["payment_value"].sum().reset_index()

# item information for each order
items_order = items.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    item_value=("price", "sum"),
    freight_value=("freight_value", "sum")
).reset_index()

# combine order, payment and item information
customer_orders = orders.merge(payment_order, on="order_id", how="left")
customer_orders = customer_orders.merge(items_order, on="order_id", how="left")

# create one row for each customer
customer_data = customer_orders.groupby("customer_id").agg(
    purchase_frequency=("order_id", "nunique"),
    total_spend=("payment_value", "sum"),
    average_order_value=("payment_value", "mean"),
    total_items=("item_count", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

customer_data.head()

## 3. Descriptive statistics of customer features

In [ ]:
customer_data.describe()

## 4. Preparing the data for K-Means

In [ ]:
features = [
    "purchase_frequency",
    "total_spend",
    "average_order_value",
    "total_items",
    "total_freight"
]

X = customer_data[features].fillna(0)

# reduce the effect of very large values
X_log = np.log1p(X)

# scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("Features used:")
print(features)

## 5. Choosing the number of clusters

In [ ]:
# Silhouette score is calculated on a sample to make the analysis faster.
sample_size = min(10000, len(customer_data))
sample_index = np.random.RandomState(42).choice(
    len(customer_data), sample_size, replace=False
)

X_sample = X_scaled[sample_index]

silhouette_scores = {}

for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_sample)

    score = silhouette_score(X_sample, labels)
    silhouette_scores[k] = score

    print("K =", k, "Silhouette Score =", round(score, 3))

plt.figure(figsize=(8, 5))
plt.plot(
    list(silhouette_scores.keys()),
    list(silhouette_scores.values()),
    marker="o"
)
plt.title("Silhouette Score for Different K Values")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.show()

## 6. Applying K-Means clustering

In [ ]:
# select the K value with the highest silhouette score
best_k = max(silhouette_scores, key=silhouette_scores.get)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
customer_data["cluster"] = kmeans.fit_predict(X_scaled)

print("Number of clusters selected:", best_k)
customer_data.head()

## 7. Visualizing the customer segments

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    customer_data["total_spend"],
    customer_data["purchase_frequency"],
    c=customer_data["cluster"],
    alpha=0.4
)

plt.title("Customer Segments")
plt.xlabel("Total Spend")
plt.ylabel("Purchase Frequency")
plt.show()

## 8. Comparing the clusters

In [ ]:
cluster_summary = customer_data.groupby("cluster")[features].mean().round(2)
cluster_summary

In [ ]:
cluster_size = customer_data["cluster"].value_counts().sort_index()

cluster_size.plot(kind="bar", figsize=(8, 5))
plt.title("Number of Customers in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

## 9. Understanding the clusters

The K-Means model divided the customers into **3 groups**.

The clusters are based on:
- Purchase frequency
- Total spending
- Average order value
- Number of items purchased
- Freight spending

The cluster number itself is only a label. To understand each segment, the average values in the cluster summary should be compared.

For example, a cluster with higher total spending and purchase frequency can be treated as a higher-value or more active customer group, while a cluster with lower activity may need re-engagement.

## 10. Possible business uses

The customer segments can be useful for:

1. **Customer retention** - identify customers who purchase frequently.
2. **Personalized offers** - create offers based on spending behaviour.
3. **Re-engagement campaigns** - target customers with lower activity.
4. **Customer loyalty programs** - provide benefits based on customer value.
5. **Marketing analysis** - study different customer groups separately.

## 11. Conclusion

Customer segmentation helps convert a large customer dataset into smaller groups with similar behaviour.

In this task, K-Means clustering was used to segment Olist customers using purchase frequency, spending, average order value, number of items and freight spending.

The resulting clusters provide useful information about different customer behaviours and can help businesses plan more targeted marketing and customer-retention strategies.